# 00b - Update the code on Drive (no dataset rebuild)
Run this when the code changed. It rewrites all `.py` files and copies them to `MyDrive/plant-disease-v3/code/`.
The dataset (`data_v3.tar`, `manifest.csv`) is NOT touched. The 01a-01d notebooks copy their code from that Drive folder at start, so re-run those afterwards.
This version adds: train accuracy per epoch, clean train/val/test accuracy in evaluate.py, pooled-only test evaluation.

In [ ]:
from google.colab import drive; drive.mount("/content/drive")
import os
os.makedirs("/content/plant_pipeline", exist_ok=True)
%cd /content/plant_pipeline

Mounted at /content/drive
/content/plant_pipeline


In [ ]:
%%writefile /content/plant_pipeline/config.py
"""Single source of truth. Every model is trained with EXACTLY these settings; only the architecture name differs."""
import os
from pathlib import Path
SEED = 42
DRIVE = Path(os.environ.get("PLANT_DRIVE", "/content/drive/MyDrive/plant-disease-v3")); WORK = Path(os.environ.get("PLANT_WORK", "/content/data_v3"))
for p in (DRIVE, WORK, DRIVE / "checkpoints", DRIVE / "results"): p.mkdir(parents=True, exist_ok=True)

ID_CLASSES = [   # 10 classes = 3 crops
    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot", "Corn_(maize)___Common_rust_",
    "Corn_(maize)___Northern_Leaf_Blight", "Corn_(maize)___healthy",
    "Grape___Black_rot", "Grape___Esca_(Black_Measles)", "Grape___Leaf_blight_(Isariopsis_Leaf_Spot)", "Grape___healthy",
    "Pepper,_bell___Bacterial_spot", "Pepper,_bell___healthy"]
K = len(ID_CLASSES)
TARGET_CROPS = ("corn", "grape", "pepper")

SPLIT = (0.70, 0.15, 0.15)       # train / val / test, stratified per (class, source), group-aware
MAX_SHORT_SIDE = 320
IMG = 224; MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
TRAIN = dict(epochs=15, batch=32, weight_decay=0.05, head_mult=10, warmup_epochs=1, label_smoothing=0.1,
             lr=1e-4,   # ONE learning rate for ALL models (classifier head uses lr x head_mult, same multiplier for all)
             field_boost=3.0)    # PlantDoc images are sampled 3x more often in training (PlantDoc is small)
COMMON_KW = dict(drop_rate=0.2, drop_path_rate=0.1)
MODELS = {
    "efficientnet_b0":   dict(timm="efficientnet_b0"),
    "mobilenetv3_large": dict(timm="mobilenetv3_large_100"),
    "dinov2_vit_s14":    dict(timm="vit_small_patch14_dinov2", kw=dict(img_size=224)),
    "convnext_tiny":     dict(timm="convnext_tiny"),
}
REJECT = dict(tpr=0.95, shrink=0.01)  # deployment-time rejection threshold = 95th pct of Mahalanobis distance on VALIDATION images

Writing /content/plant_pipeline/config.py


In [ ]:
%%writefile /content/plant_pipeline/splits.py
"""Leak-free, stratified split utilities (pure numpy/scipy, unit-testable)."""
import random
import numpy as np
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components

SPLIT_NAMES = ["train", "val", "test"]

def adaptive_groups(feats, thr_grid=(0.85, 0.88, 0.90, 0.92, 0.95), max_frac=0.05):
    """Cluster near-duplicate images (same leaf photographed repeatedly / crops of one photo).
    feats: L2-normalised (n,d). Picks the most aggressive cosine threshold that does not create a giant group."""
    n = len(feats)
    if n == 1: return np.zeros(1, int), None
    S = feats @ feats.T
    lab = np.arange(n); used = None
    for thr in thr_grid:                                   # ascending => descending group size
        _, lab = connected_components(csr_matrix(S >= thr), directed=False); used = thr
        if np.bincount(lab).max() <= max(2, max_frac * n): break
    return lab, used

def assign_groups(group_ids, ratios, seed):
    """Greedy largest-deficit assignment of whole groups to splits -> ratios hit almost exactly, no group is split."""
    rng = random.Random(seed)
    ids, sizes = np.unique(group_ids, return_counts=True)
    order = sorted(zip(ids, sizes), key=lambda t: (-t[1], rng.random()))
    target = np.array(ratios, float) * sizes.sum(); filled = np.zeros(len(ratios)); ok = np.array(ratios) > 0
    out = {}
    for g, s in order:
        deficit = np.where(ok, target - filled, -np.inf)
        i = int(np.argmax(deficit)); out[g] = i; filled[i] += s
    return np.array([out[g] for g in group_ids])

Writing /content/plant_pipeline/splits.py


In [ ]:
%%writefile /content/plant_pipeline/ood_utils.py
import numpy as np
from scipy.special import softmax
from scipy.optimize import minimize_scalar

def ece(probs, y, bins=15):
    conf, pred = probs.max(1), probs.argmax(1); acc = pred == y; e = 0.0; edges = np.linspace(0, 1, bins + 1)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return float(e)

def fit_temperature(logits, y):
    nll = lambda t: -np.log(softmax(logits / np.exp(t), 1)[np.arange(len(y)), y] + 1e-12).mean()
    return float(np.exp(minimize_scalar(nll, bounds=(-3, 3), method="bounded").x))

def fit_gaussians(feats, y, k, shrink=0.01):
    f = feats.astype(np.float64); mu = np.stack([f[y == c].mean(0) for c in range(k)]); cen = f - mu[y]
    cov = cen.T @ cen / len(cen); cov = (1 - shrink) * cov + shrink * np.trace(cov) / cov.shape[0] * np.eye(cov.shape[0])
    return mu, np.linalg.inv(cov)

def sqdist(feats, mu, P):
    f = feats.astype(np.float64); A = f @ P
    return (A * f).sum(1)[:, None] - 2 * A @ mu.T + ((mu @ P) * mu).sum(1)[None, :]

Writing /content/plant_pipeline/ood_utils.py


In [ ]:
%%writefile /content/plant_pipeline/sources.py
"""How we read the two raw datasets (shared by audit_sources.py and build_dataset.py)."""
from pathlib import Path
import config as C
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def images_in(d): return sorted(x for x in Path(d).iterdir() if x.is_file() and x.suffix.lower() in IMG_EXT)

def pv_class_dirs(pv_root): return sorted(p for p in Path(pv_root).iterdir() if p.is_dir())

def pd_class_dirs(pd_root):
    """PlantDoc-Dataset (cropped leaves) has train/<class>/ and test/<class>/. Falls back to <root>/<class>/ if the layout differs."""
    root = Path(pd_root); parts = [(p.name, root / p.name) for p in ("train", "test") if (root / p).is_dir()] if False else \
           [(n, root / n) for n in ("train", "test") if (root / n).is_dir()]
    if not parts: parts = [("all", root)]
    return [(part, d) for part, base in parts for d in sorted(x for x in base.iterdir() if x.is_dir() and not x.name.startswith("."))
            if any(f.suffix.lower() in IMG_EXT for f in d.iterdir() if f.is_file())]

def pd_map(name):
    """PlantDoc class name -> one of our 10 classes, or None (not part of this project)."""
    n = name.lower().replace("_", " ").strip(); crop = next((k for k in C.TARGET_CROPS if k in n), None); I = C.ID_CLASSES
    if crop == "corn":
        if "gray" in n: return I[0]
        if "rust" in n: return I[1]
        if "blight" in n: return I[2]
    elif crop == "grape":
        if "black rot" in n: return I[4]
        if n == "grape leaf": return I[7]
    elif crop == "pepper":
        if "spot" in n: return I[8]
        if n.endswith("leaf"): return I[9]
    return None

Writing /content/plant_pipeline/sources.py


In [ ]:
%%writefile /content/plant_pipeline/common.py
import json, hashlib, random
import numpy as np, pandas as pd, torch, timm
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import config as C

def seed_all(s=C.SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def load_manifest(): return pd.read_csv(C.WORK / "manifest.csv")
def manifest_md5(): return hashlib.md5(open(C.WORK / "manifest.csv", "rb").read()).hexdigest()

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(C.IMG, scale=(0.4, 1.0)), transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(0.3),
    transforms.RandomRotation(25), transforms.ColorJitter(0.35, 0.35, 0.3, 0.05),
    transforms.RandomApply([transforms.GaussianBlur(5, (0.1, 2.0))], p=0.3),
    transforms.ToTensor(), transforms.Normalize(C.MEAN, C.STD), transforms.RandomErasing(p=0.25, scale=(0.02, 0.15))])
eval_tf = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(C.IMG), transforms.ToTensor(), transforms.Normalize(C.MEAN, C.STD)])

class DS(Dataset):
    def __init__(self, df, tf, ycol="y"):
        self.p = [C.WORK / p for p in df.path]; self.y = df[ycol].values; self.tf = tf
    def __len__(self): return len(self.p)
    def __getitem__(self, i): return self.tf(Image.open(self.p[i]).convert("RGB")), int(self.y[i])

def make_loader(df, tf, bs=64, **kw): return DataLoader(DS(df, tf), batch_size=bs, num_workers=2, pin_memory=True, **kw)

def build_model(name, num_classes, pretrained=True):
    spec = C.MODELS[name]
    return timm.create_model(spec["timm"], pretrained=pretrained, num_classes=num_classes, **C.COMMON_KW, **spec.get("kw", {}))

@torch.inference_mode()
def extract(model, loader, dev):
    """logits, penultimate features, labels (eval mode)"""
    model.eval(); L, Fe, Y = [], [], []
    for x, y in loader:
        x = x.to(dev, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16, enabled=dev.type == "cuda"):
            f = model.forward_head(model.forward_features(x), pre_logits=True); lg = model.get_classifier()(f)
        L.append(lg.float().cpu()); Fe.append(f.float().cpu()); Y.append(y)
    return torch.cat(L).numpy(), torch.cat(Fe).numpy(), torch.cat(Y).numpy()

Writing /content/plant_pipeline/common.py


In [ ]:
%%writefile /content/plant_pipeline/audit_sources.py
"""Inspect the RAW datasets and save everything to Drive (results/audit/), so numbers are never guessed.
   python audit_sources.py --pv .../PlantVillage-Dataset/raw/color --pd .../PlantDoc-Dataset"""
import argparse, json
import numpy as np, pandas as pd
from PIL import Image
import config as C
from sources import images_in, pv_class_dirs, pd_class_dirs, pd_map

def size_stats(files, n=150):
    r = np.random.default_rng(0); ws = []
    for f in r.choice(files, min(n, len(files)), replace=False):
        try: ws.append(Image.open(f).size)
        except Exception: pass
    a = np.array(ws); return dict(median_w=int(np.median(a[:, 0])), median_h=int(np.median(a[:, 1])), min_short=int(a.min(1).min())) if len(a) else {}

def main(a):
    out = C.DRIVE / "results" / "audit"; out.mkdir(parents=True, exist_ok=True)
    pv = [dict(cls=d.name, n=len(images_in(d)), used=d.name in C.ID_CLASSES) for d in pv_class_dirs(a.pv)]
    pvd = pd.DataFrame(pv); pvd.to_csv(out / "plantvillage_all_classes.csv", index=False)
    pdl = [dict(part=p, cls=d.name, n=len(images_in(d)), mapped_to=pd_map(d.name) or "-- dropped --") for p, d in pd_class_dirs(a.pd)]
    pdd = pd.DataFrame(pdl); pdd.to_csv(out / "plantdoc_all_classes.csv", index=False)
    cov = pd.DataFrame({"plantvillage": pvd[pvd.used].set_index("cls").n.reindex(C.ID_CLASSES),
                        "plantdoc": pdd[pdd.mapped_to != "-- dropped --"].groupby("mapped_to").n.sum().reindex(C.ID_CLASSES).fillna(0).astype(int)})
    cov["plantdoc_raw_folders"] = [", ".join(sorted(set(pdd[pdd.mapped_to == c].cls))) or "-" for c in C.ID_CLASSES]
    cov.loc["TOTAL"] = [cov.plantvillage.sum(), cov.plantdoc.sum(), ""]; cov.to_csv(out / "coverage_of_our_10_classes.csv")
    pvf = [f for d in pv_class_dirs(a.pv) for f in images_in(d)[:30]]; pdf = [f for _, d in pd_class_dirs(a.pd) for f in images_in(d)[:30]]
    summ = dict(plantvillage=dict(total_images=int(pvd.n.sum()), n_classes=len(pvd), images_in_our_10_classes=int(pvd[pvd.used].n.sum()), **size_stats(pvf)),
                plantdoc=dict(total_images=int(pdd.n.sum()), n_class_folders=int(pdd.cls.nunique()), parts=pdd.groupby("part").n.sum().to_dict(),
                              images_in_our_classes=int(pdd[pdd.mapped_to != "-- dropped --"].n.sum()), **size_stats(pdf)))
    json.dump(summ, open(out / "raw_dataset_summary.json", "w"), indent=2)
    pd.set_option("display.width", 250, "display.max_rows", 200, "display.max_colwidth", 60)
    print("PlantVillage: all class folders"); print(pvd.to_string(index=False)); print("\nPlantDoc: all class folders -> how we map them"); print(pdd.to_string(index=False))
    print("\n== COVERAGE OF OUR 10 CLASSES (raw, before cleaning/splitting) =="); print(cov); print("\n", json.dumps(summ, indent=1))
    print("\nsaved to", out, "-> you can open these CSVs any time from Drive")

if __name__ == "__main__":
    ap = argparse.ArgumentParser(); ap.add_argument("--pv", required=True); ap.add_argument("--pd", required=True); main(ap.parse_args())

Writing /content/plant_pipeline/audit_sources.py


In [ ]:
%%writefile /content/plant_pipeline/build_dataset.py
"""Build the dataset + manifest (PlantVillage + PlantDoc only, 10 classes).

  python build_dataset.py --pv /content/PlantVillage-Dataset/raw/color --pd /content/PlantDoc-Dataset

Each source is split 70/15/15 per class (stratified, group-aware), then the splits are POOLED:
  train = PV_train + PD_train | val = PV_val + PD_val | test = PV_test + PD_test
"""
import argparse, hashlib, json, random, subprocess
from pathlib import Path
import numpy as np, pandas as pd
from PIL import Image
import config as C
from splits import adaptive_groups, assign_groups, SPLIT_NAMES

from sources import IMG_EXT, images_in, pv_class_dirs, pd_class_dirs, pd_map
rows = []; STATS = dict(unreadable=0, too_small=0, blank=0)

def add_file(f, rel, label, source, domain):
    try:
        with Image.open(f) as img:
            img = img.convert("RGB"); w, h = img.size; s = min(w, h)
            if s < 32: STATS["too_small"] += 1; return
            if np.asarray(img.resize((32, 32))).std() < 3: STATS["blank"] += 1; return
            dst = C.WORK / rel; dst.parent.mkdir(parents=True, exist_ok=True)
            if s > C.MAX_SHORT_SIDE: k = C.MAX_SHORT_SIDE / s; img = img.resize((round(w * k), round(h * k)), Image.BICUBIC)
            img.save(dst, "JPEG", quality=92)
        rows.append(dict(path=rel, label=label, source=source, domain=domain))
    except Exception as e: STATS["unreadable"] += 1; print("skip", f, e)

def load_pv(pv):                                   # lab images; only the 10 chosen class folders are used
    for name in C.ID_CLASSES:
        d = Path(pv) / name; assert d.is_dir(), f"missing PlantVillage folder {name}"
        for f in images_in(d): add_file(f, f"pv/{name}/{f.name}", name, "plantvillage", "lab")

def load_pd(root):                                 # real-world images; official train+test merged, we re-split ourselves
    for part, d in pd_class_dirs(root):
        lab = pd_map(d.name)
        if lab is None: continue                   # other crops / unmapped classes are not part of this project
        for f in images_in(d): add_file(f, f"pd/{part}_{d.name}/{f.name}", lab, "plantdoc", "field")

# ------------------------------ embeddings -> groups ------------------------------
def embed(paths):
    import torch, timm
    from torchvision import transforms
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    m = timm.create_model("vit_small_patch14_dinov2", pretrained=True, num_classes=0, img_size=224).to(dev).eval()
    tf = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(), transforms.Normalize(C.MEAN, C.STD)])
    out = []
    with torch.inference_mode(), torch.autocast(dev.type, enabled=dev.type == "cuda"):
        for i in range(0, len(paths), 64):
            x = torch.stack([tf(Image.open(C.WORK / p).convert("RGB")) for p in paths[i:i + 64]]).to(dev)
            out.append(m(x).float().cpu().numpy())
    f = np.concatenate(out); return f / np.linalg.norm(f, axis=1, keepdims=True)

def main(a):
    load_pv(a.pv); load_pd(a.pd)
    df = pd.DataFrame(rows)
    df["md5"] = [hashlib.md5(open(C.WORK / p, "rb").read()).hexdigest() for p in df.path]
    n0 = len(df); df = df.drop_duplicates("md5").reset_index(drop=True); STATS["exact_duplicates_removed"] = n0 - len(df); STATS["images_before_cleaning_dedup"] = n0
    print("cleaning:", STATS)
    df["split"] = ""; df["group"] = ""
    for (source, label), g in df.groupby(["source", "label"]):
        lab, thr = adaptive_groups(embed(g.path.tolist()))
        s = assign_groups(lab, C.SPLIT, C.SEED); idx = g.index.values
        df.loc[idx, "split"] = [SPLIT_NAMES[i] for i in s]; df.loc[idx, "group"] = [f"{source}|{label}|{k}" for k in lab]
        print(f"{source:12s} {label[:40]:40s} n={len(g):5d} groups={len(set(lab)):5d} sim_thr={thr}")
    df["y"] = df.label.map({c: i for i, c in enumerate(C.ID_CLASSES)}).astype(int)
    assert (df.groupby("group").split.nunique() == 1).all(), "a group straddles splits -> leakage"

    pd.set_option("display.width", 250, "display.max_rows", 100)
    tab = pd.crosstab(df.label, [df.split, df.source]).reindex(C.ID_CLASSES)
    tab.to_csv(C.DRIVE / "results" / "dataset_class_split_source.csv")
    print("\n== images per class x split x source =="); print(tab); print("\n== totals =="); print(pd.crosstab(df.source, df.split, margins=True))
    if (pd.crosstab(df[df.source == "plantdoc"].label, df[df.source == "plantdoc"].split) < 5).any().any():
        print("\n!! some PlantDoc class has < 5 images in a split -> PlantDoc numbers are noisy; use the confidence intervals")
    miss = set(C.ID_CLASSES) - set(df[df.source == "plantdoc"].label); print("classes with NO PlantDoc images:", sorted(miss))

    df.to_csv(C.WORK / "manifest.csv", index=False); df.to_csv(C.DRIVE / "manifest.csv", index=False)
    json.dump(dict(class_names=C.ID_CLASSES, seed=C.SEED, split=C.SPLIT, cleaning=STATS, total_images=len(df),
                   by_source={k: int(v) for k, v in df.source.value_counts().items()},
                   by_split_source={f"{a}|{b}": int(v) for (a, b), v in df.groupby(["split", "source"]).size().items()}),
              open(C.DRIVE / "meta.json", "w"), indent=2)
    subprocess.run(["tar", "-cf", str(C.DRIVE / "data_v3.tar"), "-C", str(C.WORK), "."], check=True); print("dataset saved to Drive")

if __name__ == "__main__":
    ap = argparse.ArgumentParser(); ap.add_argument("--pv", required=True); ap.add_argument("--pd", required=True); main(ap.parse_args())

Writing /content/plant_pipeline/build_dataset.py


In [ ]:
%%writefile /content/plant_pipeline/train.py
"""Identical protocol for every model.
  python train.py --model efficientnet_b0      # every model uses the SAME lr / epochs / optimizer / schedule / augmentation from config.py
"""
import argparse, json, math, time
import numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import WeightedRandomSampler, DataLoader
from sklearn.metrics import f1_score, accuracy_score
import config as C
from common import *

def fit(name, lr, epochs, tr, va, ncls, ckpt=None, dev=torch.device("cuda")):
    T = C.TRAIN; seed_all()
    model = build_model(name, ncls).to(dev)
    head_ids = {id(p) for p in model.get_classifier().parameters()}
    head = [p for p in model.parameters() if id(p) in head_ids]; body = [p for p in model.parameters() if id(p) not in head_ids]
    opt = torch.optim.AdamW([{"params": body, "lr": lr}, {"params": head, "lr": lr * T["head_mult"]}], weight_decay=T["weight_decay"])
    w = np.where(tr.domain.values == "field", T["field_boost"], 1.0)                      # oversample real-world images
    loader = DataLoader(DS(tr, train_tf), batch_size=T["batch"], sampler=WeightedRandomSampler(w, len(tr), replacement=True),
                        num_workers=2, pin_memory=True, drop_last=True)
    vl = make_loader(va, eval_tf)
    cnt = np.bincount(tr.y.values, minlength=ncls).astype(float); cw = torch.tensor(cnt.sum() / (ncls * np.maximum(cnt, 1)), dtype=torch.float32).to(dev)
    crit = nn.CrossEntropyLoss(weight=cw, label_smoothing=T["label_smoothing"])
    total = epochs * len(loader); warm = T["warmup_epochs"] * len(loader)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else 0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    scaler = torch.amp.GradScaler("cuda"); best, best_ep, hist = -1, 0, []
    for ep in range(1, epochs + 1):                                                         # NO early stopping: every model sees `epochs` epochs
        model.train(); t0 = time.time(); tl = n = tc = 0
        for x, y in loader:
            x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True); opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.float16): out = model(x); loss = crit(out, y)
            scaler.scale(loss).backward(); scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step(); tl += loss.item() * len(y); n += len(y); tc += (out.argmax(1) == y).sum().item()
        lg, _, yv = extract(model, vl, dev); p = lg.argmax(1)
        f1 = f1_score(yv, p, average="macro", zero_division=0); acc = accuracy_score(yv, p)
        hist.append(dict(epoch=ep, train_loss=tl / n, train_acc=tc / n, val_acc=acc, val_f1=f1))
        star = ""
        if f1 > best:
            best, best_ep, star = f1, ep, " *"
            if ckpt: torch.save(dict(model=name, num_classes=ncls, lr=lr, epochs=epochs, epoch=ep, state=model.state_dict()), ckpt)
        print(f"[{name} lr={lr:g}] ep {ep:02d}/{epochs} {time.time()-t0:4.0f}s loss {tl/n:.3f} train acc {tc/n:.4f} | val acc {acc:.4f} F1 {f1:.4f}{star}")
    return best, best_ep, hist

def main(a):
    dev = torch.device("cuda"); assert torch.cuda.is_available(), "use a GPU runtime"
    df = load_manifest(); ncls = C.K
    tr, va = df[df.split == "train"].copy(), df[df.split == "val"].copy()
    tag = a.model
    print(f"train {len(tr)} | val {len(va)} | classes {ncls} | md5 {manifest_md5()[:8]}")
    T = C.TRAIN; lr = T["lr"]; sweep = None
    best, best_ep, hist = fit(a.model, lr, T["epochs"], tr, va, ncls, C.DRIVE / "checkpoints" / f"{tag}.pth", dev)
    json.dump(dict(tag=tag, model=a.model, lr=lr, lr_sweep={str(k): v for k, v in (sweep or {}).items()}, epochs=T["epochs"],
                   best_epoch=best_ep, best_val_f1=best, manifest_md5=manifest_md5(), history=hist),
              open(C.DRIVE / "results" / f"{tag}_train.json", "w"), indent=2)

if __name__ == "__main__":
    ap = argparse.ArgumentParser(); ap.add_argument("--model", required=True, choices=list(C.MODELS));     main(ap.parse_args())

Writing /content/plant_pipeline/train.py


In [ ]:

# Add per-epoch training and validation recall to train.py

from pathlib import Path

train_file = Path("/content/plant_pipeline/train.py")
source = train_file.read_text()

old_import = "from sklearn.metrics import f1_score, accuracy_score"
new_import = (
    "from sklearn.metrics import "
    "f1_score, accuracy_score, recall_score"
)

old_loop = """model.train(); t0 = time.time(); tl = n = tc = 0
        for x, y in loader:"""

new_loop = """model.train(); t0 = time.time(); tl = n = tc = 0
        train_y_all, train_pred_all = [], []
        for x, y in loader:"""

old_batch = """scaler.step(opt); scaler.update(); sched.step(); tl += loss.item() * len(y); n += len(y); tc += (out.argmax(1) == y).sum().item()"""

new_batch = """scaler.step(opt); scaler.update(); sched.step()
            tl += loss.item() * len(y)
            n += len(y)
            batch_pred = out.detach().argmax(1)
            tc += (batch_pred == y).sum().item()
            train_y_all.extend(y.detach().cpu().tolist())
            train_pred_all.extend(batch_pred.cpu().tolist())"""

old_history = """lg, _, yv = extract(model, vl, dev); p = lg.argmax(1)
        f1 = f1_score(yv, p, average="macro", zero_division=0); acc = accuracy_score(yv, p)
        hist.append(dict(epoch=ep, train_loss=tl / n, train_acc=tc / n, val_acc=acc, val_f1=f1))"""

new_history = """lg, _, yv = extract(model, vl, dev); p = lg.argmax(1)
        f1 = f1_score(yv, p, average="macro", zero_division=0)
        acc = accuracy_score(yv, p)

        train_recall = recall_score(
            train_y_all, train_pred_all,
            average="macro", zero_division=0
        )
        val_recall = recall_score(
            yv, p, average="macro", zero_division=0
        )

        hist.append(dict(
            epoch=ep,
            train_loss=tl / n,
            train_acc=tc / n,
            train_recall=train_recall,
            val_acc=acc,
            val_recall=val_recall,
            val_f1=f1
        ))"""

replacements = [
    (old_import, new_import),
    (old_loop, new_loop),
    (old_batch, new_batch),
    (old_history, new_history),
]

for old, new in replacements:
    if old not in source:
        raise ValueError(
            "Expected training code was not found. "
            "Do not run training until train.py is checked."
        )
    source = source.replace(old, new, 1)

train_file.write_text(source)

# Update the copy stored on Google Drive
import shutil
drive_code = Path("/content/drive/MyDrive/plant-disease-v3/code")
drive_code.mkdir(parents=True, exist_ok=True)
shutil.copy2(train_file, drive_code / "train.py")

print("Updated train.py successfully.")
print("Training and validation macro-recall will be saved per epoch.")


Updated train.py successfully.
Training and validation macro-recall will be saved per epoch.


In [ ]:
%%writefile /content/plant_pipeline/evaluate.py
"""Test-set evaluation on the single POOLED test set + deployment reject threshold.
   python evaluate.py --tag efficientnet_b0
Nothing here is tuned on the test set: temperature + reject threshold come from VALIDATION images only."""
import argparse, json
import numpy as np, pandas as pd, torch
from sklearn.metrics import f1_score, precision_recall_fscore_support, confusion_matrix
import config as C
from common import *
from ood_utils import *

def boot_ci(ok, n=1000):
    r = np.random.default_rng(0); b = [r.choice(ok, len(ok)).mean() for _ in range(n)]; return [float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5))]

def main(a):
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ck = torch.load(C.DRIVE / "checkpoints" / f"{a.tag}.pth", map_location=dev, weights_only=False)
    model = build_model(ck["model"], C.K, pretrained=False).to(dev); model.load_state_dict(ck["state"]); model.eval()
    df = load_manifest(); S = {k: df[df.split == k] for k in ("train", "val", "test")}
    O = {k: extract(model, make_loader(v, eval_tf), dev) for k, v in S.items()}
    T = fit_temperature(O["val"][0], O["val"][2])
    lg, ft, y = O["test"]; pred = lg.argmax(1)               # ONE pooled test set (PlantVillage-test + PlantDoc-test together)
    res = dict(tag=a.tag, model=ck["model"], lr=ck["lr"], epochs=ck["epochs"], best_epoch=ck["epoch"], temperature=T, manifest_md5=manifest_md5(),
               params_m=sum(p.numel() for p in model.parameters()) / 1e6)
    ok = (pred == y).astype(float)
    res["classification"] = dict(pooled=dict(n=int(len(y)), acc=float(ok.mean()), acc_ci95=boot_ci(ok), macro_f1=float(f1_score(y, pred, average="macro", zero_division=0))))
    res["acc_by_split_clean"] = {k: float((O[k][0].argmax(1) == O[k][2]).mean()) for k in ("train", "val", "test")}   # all with eval transforms (no augmentation)
    res["ece_before"] = ece(softmax(lg, 1), y); res["ece_after"] = ece(softmax(lg / T, 1), y)
    p, r, f, sup = precision_recall_fscore_support(y, pred, labels=range(C.K), zero_division=0)
    pd.DataFrame(dict(cls=C.ID_CLASSES, precision=p, recall=r, f1=f, support=sup)).to_csv(C.DRIVE / "results" / f"{a.tag}_per_class.csv", index=False)
    np.save(C.DRIVE / "results" / f"{a.tag}_confusion_test.npy", confusion_matrix(y, pred, labels=range(C.K)))
    # deployment-time rejection (NOT evaluated here): Mahalanobis distance, threshold from validation images
    mu, P = fit_gaussians(O["train"][1], O["train"][2], C.K, C.REJECT["shrink"])
    thr = float(np.quantile(sqdist(O["val"][1], mu, P).min(1), C.REJECT["tpr"]))
    json.dump(dict(temperature=T, threshold=thr, class_names=C.ID_CLASSES), open(C.DRIVE / "checkpoints" / f"{a.tag}_reject.json", "w"))
    np.savez(C.DRIVE / "checkpoints" / f"{a.tag}_reject_stats.npz", mu=mu, P=P)
    json.dump(res, open(C.DRIVE / "results" / f"{a.tag}_final.json", "w"), indent=2)
    v = res["classification"]["pooled"]; print(f"POOLED TEST n={v['n']} acc {v['acc']*100:.2f}% (95% CI {[round(x*100,1) for x in v['acc_ci95']]}) macro-F1 {v['macro_f1']*100:.2f}%")
    print("clean accuracy (no augmentation):", {k: round(v * 100, 2) for k, v in res["acc_by_split_clean"].items()})
    print(f"ECE {res['ece_before']:.4f} -> {res['ece_after']:.4f} after temperature T={T:.2f}")

if __name__ == "__main__":
    ap = argparse.ArgumentParser(); ap.add_argument("--tag", required=True); main(ap.parse_args())

Writing /content/plant_pipeline/evaluate.py


In [ ]:
%%writefile /content/plant_pipeline/compare.py
"""python compare.py  -> fair comparison table + protocol audit"""
import json, glob
import pandas as pd, matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
import config as C
R = [json.load(open(f)) for f in sorted(glob.glob(str(C.DRIVE / "results" / "*_final.json")))]; assert R, "run evaluate.py first"
assert len({r["manifest_md5"] for r in R}) == 1, "models were trained on DIFFERENT dataset builds"
assert len({r["epochs"] for r in R}) == 1, "epochs differ between models"
print("protocol audit OK: same split, epochs =", R[0]["epochs"], "| LR:", {r["tag"]: r["lr"] for r in R})
rows = [{"model": r["tag"], "params(M)": round(r["params_m"], 1), "lr": r["lr"], "epochs": r["epochs"], "best epoch": r["best_epoch"],
         "test acc %": round(r["classification"]["pooled"]["acc"] * 100, 2), "95% CI": [round(x * 100, 1) for x in r["classification"]["pooled"]["acc_ci95"]],
         "macro-F1 %": round(r["classification"]["pooled"]["macro_f1"] * 100, 2), "ECE after T": round(r["ece_after"], 4)} for r in R]
T = pd.DataFrame(rows).set_index("model"); pd.set_option("display.width", 250, "display.max_columns", 30); print(T.T); T.to_csv(C.DRIVE / "results" / "final_comparison.csv")
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for a, c in zip(ax, ["test acc %", "macro-F1 %"]): T[c].plot.bar(ax=a, color="#4c78a8"); a.set_title(c); a.tick_params(axis="x", rotation=25)
plt.tight_layout(); plt.savefig(C.DRIVE / "results" / "final_comparison.png", dpi=140)

Writing /content/plant_pipeline/compare.py


In [ ]:
%%writefile /content/plant_pipeline/predict.py
"""python predict.py --tag convnext_tiny img1.jpg img2.jpg   (prints top-3, or REJECTED if too far from every known class)"""
import argparse, json
import numpy as np, torch
from PIL import Image
import config as C
from common import build_model, eval_tf
from ood_utils import sqdist
from scipy.special import softmax
ap = argparse.ArgumentParser(); ap.add_argument("--tag", required=True); ap.add_argument("imgs", nargs="+"); a = ap.parse_args()
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ck = torch.load(C.DRIVE / "checkpoints" / f"{a.tag}.pth", map_location=dev, weights_only=False)
m = build_model(ck["model"], C.K, pretrained=False).to(dev); m.load_state_dict(ck["state"]); m.eval()
th = json.load(open(C.DRIVE / "checkpoints" / f"{a.tag}_reject.json")); st = np.load(C.DRIVE / "checkpoints" / f"{a.tag}_reject_stats.npz")
for p in a.imgs:
    x = eval_tf(Image.open(p).convert("RGB")).unsqueeze(0).to(dev)
    with torch.inference_mode(): f = m.forward_head(m.forward_features(x), pre_logits=True); lg = m.get_classifier()(f)
    d = float(sqdist(f.float().cpu().numpy(), st["mu"], st["P"]).min()); pr = softmax(lg.float().cpu().numpy() / th["temperature"], 1)[0]
    rej = d > th["threshold"]; print(f"\n{p}\n  Mahalanobis {d:.1f} (threshold {th['threshold']:.1f}) -> " + ("REJECTED: not a supported leaf" if rej else "ACCEPTED"))
    if not rej:
        for i in pr.argsort()[::-1][:3]: print(f"   {pr[i]*100:5.1f}%  {C.ID_CLASSES[i]}")

Writing /content/plant_pipeline/predict.py


In [ ]:
!mkdir -p /content/drive/MyDrive/plant-disease-v3/code
!cp /content/plant_pipeline/*.py /content/drive/MyDrive/plant-disease-v3/code/
!ls -la /content/drive/MyDrive/plant-disease-v3/code
!grep -c "train acc" /content/drive/MyDrive/plant-disease-v3/code/train.py

total 38
drwx------ 2 root root 4096 Oct 26 12:44 .
drwx------ 3 root root 4096 Oct 26 12:00 ..
-rw------- 1 root root 3016 Oct 26 12:44 audit_sources.py
-rw------- 1 root root 5516 Oct 26 12:44 build_dataset.py
-rw------- 1 root root 2255 Oct 26 12:44 common.py
-rw------- 1 root root 1479 Oct 26 12:44 compare.py
-rw------- 1 root root 1841 Oct 26 12:44 config.py
-rw------- 1 root root 3421 Oct 26 12:44 evaluate.py
-rw------- 1 root root 1086 Oct 26 12:44 ood_utils.py
-rw------- 1 root root 1469 Oct 26 12:44 predict.py
-rw------- 1 root root 1616 Oct 26 12:44 sources.py
-rw------- 1 root root 1571 Oct 26 12:44 splits.py
-rw------- 1 root root 4116 Oct 26 12:44 train.py
1
